# Q1 · Part A — Derivation of the gradient and the gradient-descent update rules
**Course:** Programming for AI · **Exam:** Mid-Term Lab (Open-Book)

> **Run order for Question 1:** `01_Q1_Derivation.ipynb` → `02_Q1_Implementation.ipynb` → `03_Q1_Experiments.ipynb`

This notebook contains the **typed** derivation (so you can follow it while writing the **handwritten** version
that must be scanned for the report) and a **numerical gradient check** that proves the derived formulas are correct.

## 1. Setting up

Hypothesis (with the convention $x_0^{(i)} = 1$ for every example):

$$h_\theta(x)=\theta_0+\theta_1x_1+\dots+\theta_nx_n=\sum_{j=0}^{n}\theta_jx_j=\theta^{T}x$$

Regularised cost function ($m$ training examples, $\theta_0$ is **not** regularised):

$$J(\theta)=\frac{1}{2m}\Big[\sum_{i=1}^{m}\big(h_\theta(x^{(i)})-y^{(i)}\big)^2+\lambda\sum_{j=1}^{n}\theta_j^2\Big]$$

## 2. Three small building blocks

| Step | Result | Why |
|---|---|---|
| (a) | $\dfrac{\partial h_\theta(x^{(i)})}{\partial\theta_j}=x_j^{(i)}$ | $h$ is linear in $\theta$; only the term $\theta_jx_j$ depends on $\theta_j$ |
| (b) | $\dfrac{\partial}{\partial\theta_j}\big(h_\theta(x^{(i)})-y^{(i)}\big)^2=2\big(h_\theta(x^{(i)})-y^{(i)}\big)\,x_j^{(i)}$ | chain rule: $\frac{d}{du}u^2=2u$ with $u=h-y$ and (a) |
| (c) | $\dfrac{\partial}{\partial\theta_j}\Big(\lambda\sum_{k=1}^{n}\theta_k^2\Big)=\begin{cases}0 & j=0\\ 2\lambda\theta_j & j\ge1\end{cases}$ | in the sum only the term $k=j$ depends on $\theta_j$; $\theta_0$ is not in the sum |

## 3. Partial derivatives

### Case $j=0$ (the intercept — no regularisation)

$$\frac{\partial J}{\partial\theta_0}=\frac{1}{2m}\Big[\sum_{i=1}^{m}2\big(h_\theta(x^{(i)})-y^{(i)}\big)x_0^{(i)}+0\Big]
=\boxed{\frac{1}{m}\sum_{i=1}^{m}\big(h_\theta(x^{(i)})-y^{(i)}\big)}\qquad(\text{because }x_0^{(i)}=1)$$

### Case $j\ge1$ (regularised weights)

$$\frac{\partial J}{\partial\theta_j}=\frac{1}{2m}\Big[\sum_{i=1}^{m}2\big(h_\theta(x^{(i)})-y^{(i)}\big)x_j^{(i)}+2\lambda\theta_j\Big]
=\boxed{\frac{1}{m}\Big[\sum_{i=1}^{m}\big(h_\theta(x^{(i)})-y^{(i)}\big)x_j^{(i)}+\lambda\theta_j\Big]}$$

## 4. Gradient-descent update rules

Gradient descent moves every parameter a small step ($\alpha$ = learning rate) against its partial derivative,
**updating all $\theta_j$ simultaneously**:

$$\theta_j:=\theta_j-\alpha\frac{\partial J}{\partial\theta_j}$$

### Component form

$$\theta_0:=\theta_0-\frac{\alpha}{m}\sum_{i=1}^{m}\big(h_\theta(x^{(i)})-y^{(i)}\big)$$

$$\theta_j:=\theta_j-\frac{\alpha}{m}\Big[\sum_{i=1}^{m}\big(h_\theta(x^{(i)})-y^{(i)}\big)x_j^{(i)}+\lambda\theta_j\Big]
\;=\;\theta_j\Big(1-\frac{\alpha\lambda}{m}\Big)-\frac{\alpha}{m}\sum_{i=1}^{m}\big(h_\theta(x^{(i)})-y^{(i)}\big)x_j^{(i)}\qquad(j\ge1)$$

The factor $\big(1-\frac{\alpha\lambda}{m}\big)<1$ shrinks each weight a little on every step — this is why L2 regularisation is also called **weight decay**.

### Vector form

Write $X\in\mathbb{R}^{m\times(n+1)}$ (first column all ones), $y\in\mathbb{R}^{m}$ and $\tilde\theta=[0,\theta_1,\dots,\theta_n]^T$
(i.e. $\theta$ with the intercept replaced by 0). Then

$$J(\theta)=\frac{1}{2m}\Big[(X\theta-y)^T(X\theta-y)+\lambda\,\theta^TI'\theta\Big],\qquad I'=\mathrm{diag}(0,1,\dots,1)$$

$$\nabla_\theta J=\frac{1}{m}\Big[X^T(X\theta-y)+\lambda\tilde\theta\Big]$$

$$\boxed{\theta:=\theta-\frac{\alpha}{m}\Big[X^T(X\theta-y)+\lambda\tilde\theta\Big]}$$

Row $j$ of this vector equation is exactly the component form above (row 0 has no $\lambda$ term because $\tilde\theta_0=0$).

### Bonus — closed form (used later to verify gradient descent)
Setting $\nabla_\theta J=0$ gives the regularised normal equation $\theta=(X^TX+\lambda I')^{-1}X^Ty$.

## 5. Numerical proof that the formulas are right (gradient check)
We compare our analytic gradient with a numerical one computed from the **definition** of the derivative
(central differences, $\frac{J(\theta+\epsilon e_j)-J(\theta-\epsilon e_j)}{2\epsilon}$). If the derivation is right the two must agree to ~1e-8.

In [1]:
# ---- Project setup: finds the project root automatically ----
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def find_root():
    start = Path(os.environ.get("LAB_ROOT", Path.cwd())).resolve()
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Project root not found. Open this notebook from inside the Prog_for_AI_Lab folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

Project root: C:\Users\hp\Desktop\Prog_for_AI_Lab\Prog_for_AI_Lab


In [2]:
import numpy as np, pandas as pd
from linreg_scratch import cost_function, gradient, add_bias

In [ ]:
rng = np.random.default_rng(0)
m, n = 50, 4
X = add_bias(rng.normal(size=(m, n)))          # m x (n+1) with first column of ones
y = rng.normal(size=m)
theta = rng.normal(size=n + 1)
lam, eps = 2.5, 1e-6

analytic = gradient(X, y, theta, lam)

numeric = np.zeros_like(theta)
for j in range(len(theta)):
    e = np.zeros_like(theta); e[j] = eps
    numeric[j] = (cost_function(X, y, theta + e, lam) - cost_function(X, y, theta - e, lam)) / (2 * eps)

print(pd.DataFrame({"analytic": analytic, "numeric": numeric, "abs diff": np.abs(analytic - numeric)},
                   index=[f"theta_{j}" for j in range(n + 1)]))
print("\nMax absolute difference:", np.max(np.abs(analytic - numeric)))
assert np.allclose(analytic, numeric, atol=1e-6), "Gradient check FAILED"
print("Gradient check PASSED ✔")

         analytic   numeric      abs diff
theta_0  0.612584  0.612584  8.526158e-11
theta_1 -0.219403 -0.219403  1.078478e-10
theta_2  0.947955  0.947955  4.653933e-11
theta_3 -0.929393 -0.929393  1.728975e-10
theta_4  0.632794  0.632794  2.980058e-10

Max absolute difference: 2.980058422252796e-10
Gradient check PASSED ✔


: 

## 6. What you must do manually
Write Sections 1–4 **by hand** (clearly, every step, on paper), scan or photograph the pages, and save the images in
`reports/handwritten_q1/` (e.g. `page1.jpg`, `page2.jpg`). The report builder will insert them automatically.